## RetrievalQA

In [17]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma, FAISS
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.chains import RetrievalQA

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=200,
#     chunk_overlap=50,
# )

loader = UnstructuredFileLoader("./files/chapter_one.txt")

# docs = loader.load()
# splitter.split_documents(docs)
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # map_reduce, map_rerank...
    retriever=vectorstore.as_retriever()
)

chain.run("Describe Victory Mansions")

'Victory Mansions is the bleak, run-down block where Winston lives. The entry is through glass doors into a grim hallway that smells of boiled cabbage and old rag mats. A huge poster of Big Brother looms on the wall, its eyes seeming to follow you. The lift hardly ever works, and during daylight hours the electricity is cut off as part of an economy drive. Winston climbs seven flights of stairs to reach his flat, a small, spartan space with a telescreen in the living room. The whole building exudes deprivation and the omnipresent sense of a society under constant surveillance.'

## Stuff LCEL

In [18]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma, FAISS
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredFileLoader("./files/chapter_one.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    ("human", "{question}")
])

chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

chain.invoke("Describe Victory Mansions")

AIMessage(content='- Victory Mansions is the apartment building where Winston Smith lives, a dreary London block reached by seven flights of stairs.\n- The hallway near the entrance smells of boiled cabbage and old rag mats.\n- There is a large poster near the lift shaft, showing a huge face with a heavy moustache; the caption reads “BIG BROTHER IS WATCHING YOU.”\n- The lift is rarely working, and during daylight hours the electric current is cut off as part of an economy drive.\n- Winston has to ascend the stairs because the lift is unreliable; the building is part of the oppressive cityscape described in the text.')

## Map Reduce LCEL

In [15]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredFileLoader("./files/chapter_one.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

map_doc_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Use the following portiono of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim.
        ------
        {context}
        """,
    ),
    ("human", "{question}")    
])
map_doc_chain = map_doc_prompt | llm

def map_docs(inputs):
    documents = inputs['documents']
    question = inputs['question']
    # results = []

    # for document in documents:
    #     result = map_doc_chain.invoke({
    #         "context": document.page_content,
    #         "question": question
    #     }).content
    #     results.append(result)

    # results = "\n\n".join(results)

    # return results

    return "\n\n".join(
        map_doc_chain.invoke(
            {"context": document.page_content, "question": question}
        ).content
        for document in documents
    )
    
map_chain = {"documents": retriever, "question": RunnablePassthrough()} | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Given the following extracted parts of a long document and a question, create a final answer. 
        If you don't know the answer, just say that you don't know. Don't try to make up an answer.
        ------
        {context}
        """,
    ),
    ("human", "{question}")
])

chain = { "context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("Describe Victory Mansions")

AIMessage(content='The provided excerpt does not give a detailed physical description of Victory Mansions. It only notes that Victory Mansions is a tall building from which you could see the four Ministries, emphasizing its location and its role in the regime’s architecture. It also indicates a dreary, cramped living environment typical of the era (e.g., a lift that’s often not working, a cramped flat, and a telescreen in the room), but it doesn’t describe the building’s exterior in detail.\n\nIn the broader novel, Victory Mansions is depicted as a bleak, gray, decaying state housing block in Airstrip One (formerly Britain). The flats are small and sparsely furnished, the corridors are plain and poorly maintained, the lifts are unreliable, and the overall atmosphere is cold, surveilled, and deprived.\n\nIf you’d like, I can pull in more context from a specific passage or summarize any part of the book you’re interested in.')

In [17]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    tiktoken_model_name="gpt-3.5-turbo",
    temperature=1
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredFileLoader("./files/document.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

memory = ConversationSummaryBufferMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

def load_memory(_):
    return memory.load_memory_variables({})["chat_history"]

chain = {"context": retriever, "question": RunnablePassthrough(), "chat_history": RunnableLambda(load_memory)} | prompt | llm

def invoke_chain(question):
    result = chain.invoke(question)
    memory.save_context(
        {"input": question},
        {"output": result.content}
    )
    print(result)

invoke_chain("Is Aaronson guilty?")

content='Yes. The text states that “Jones, Aaronson, and Rutherford were guilty of the crimes they were charged with.” However, it also notes that such guilt is part of a manipulated past (a photograph proving their guilt never existed, and memories that may be false).'


In [18]:
invoke_chain("What message did he write in the table?")

content='He wrote three lines:\n- FREEDOM IS SLAVERY\n- TWO AND TWO MAKE FIVE\n- GOD IS POWER'


In [19]:
invoke_chain("Who is Julia?")

content="Julia is Winston's lover—the woman he loves and with whom he is having a forbidden relationship against the Party."
